In [ ]:
## Finetuning Script 

In [55]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from datasets import load_dataset, load_from_disk
from peft import LoraConfig, get_peft_model, TaskType
import pandas as pd
import json
import os

In [56]:
model_name = "meta-llama/Llama-3.1-8B"
local_model_dir = "./llama-base"

In [57]:
# Load from local if exists, else download and save
if os.path.exists(local_model_dir):
    print("✅ Loading model and tokenizer from local directory...")
    tokenizer = AutoTokenizer.from_pretrained(local_model_dir, trust_remote_code=True)
    # model = AutoModelForCausalLM.from_pretrained(local_model_dir, trust_remote_code=True)
else:
    print("⬇️ Downloading model and tokenizer from Hugging Face...")
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    # model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True)

    tokenizer.save_pretrained(local_model_dir)
    # model.save_pretrained(local_model_dir)

# Ensure pad token is set for training
tokenizer.pad_token = tokenizer.eos_token

✅ Loading model and tokenizer from local directory...


In [58]:
def generate_training_prompt(lhs, rhs, is_approx):
    if isinstance(lhs, str):
        lhs = [lhs] if ',' not in lhs else lhs.split(',')
        lhs_str = ", ".join(lhs)


    prompt = f"""
    You are an expert in relational database theory and data profiling.


    You are given a candidate functional dependency (FD) of the form:


    LHS → RHS


    where:
    - LHS is a set of column names from a table
    - RHS is a single column name from the same table


    A functional dependency holds if, for every pair of rows in the table, whenever the values of all columns in the LHS are equal, the value of the RHS is also equal.


    Task: Determine whether the following candidate functional dependency is genuine and holds for typical real-world data. A genuine functional dependency is one where the LHS columns meaningfully and logically determine the RHS based on domain knowledge and business rules. It represents a stable, expected relationship that should hold across real-world instances of the data, not merely a statistical coincidence of the current dataset.


    Candidate functional dependency:
    {{{lhs_str}}} → {rhs}


    You are also provided with the following information to help you make your decision:
    - Approximate FD: {is_approx}


    Rules:
    - Answer strictly with **'1'** if the LHS functionally determines the RHS (genuine).
    - Answer strictly with **'0'** if it does NOT (non-genuine).
    - Do NOT explain your reasoning.
    - Do NOT output anything other than a single character: '1' or '0'.
    - If you are uncertain, answer **'0'**.


    Answer:
    """
    return prompt.strip()

In [59]:
def format_for_training(example):
    return {
        "text": generate_training_prompt(
            lhs=example["lhs"],
            rhs=example["rhs"],
            is_approx=example["is_approx"]
        ),
        "labels": int(example["is_genuine"])
    }


In [60]:
def convert_df_to_jsonl(df, output_file):
    with open(output_file, 'w') as f:
        for _, row in df.iterrows():
            item = {
                "lhs": row["lhs"],
                "rhs": row["rhs"],
                "is_approx": row["is_approx"],
                "is_genuine": row["is_genuine"]
            }
            f.write(json.dumps(item) + "\n")


In [61]:
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    task_type=TaskType.CAUSAL_LM,
    lora_dropout=0.1,
    bias="none",
    target_modules=["c_proj", "q_proj", "v_proj"]
)
model = get_peft_model(model, peft_config)

NameError: name 'model' is not defined

In [62]:
from sklearn.model_selection import train_test_split


df = pd.read_csv("dataset.csv")
train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)

# Save for reference
train_df.to_csv("train.csv", index=False)
val_df.to_csv("val.csv", index=False)

In [64]:
convert_df_to_jsonl(train_df, "train.jsonl")
convert_df_to_jsonl(val_df, "val.jsonl")

# Load and map
data = {
    "train": load_dataset("json", data_files="train.jsonl")["train"],
    "validation": load_dataset("json", data_files="val.jsonl")["train"]
}

data["train"] = data["train"].map(format_for_training)
data["validation"] = data["validation"].map(format_for_training)

data = {
    k: v.map(lambda e: tokenizer(e["text"], padding="max_length", truncation=True, max_length=512), batched=True)
    for k, v in data.items()
}

# Add labels if needed
data = {
    k: v.map(lambda e: {"labels": e["input_ids"]}, batched=True)
    for k, v in data.items()
}

data["train"].save_to_disk("qwen-fd-finetuned/train_dataset")
data["validation"].save_to_disk("qwen-fd-finetuned/val_dataset")

Generating train split: 511 examples [00:00, 47287.13 examples/s]
Generating train split: 57 examples [00:00, 4807.08 examples/s]
Saving the dataset (1/1 shards): 100%|██████████| 57/57 [00:00<00:00, 9908.22 examples/s] 


In [65]:
data = {
    "train": load_from_disk("qwen-fd-finetuned/train_dataset"),
    "validation": load_from_disk("qwen-fd-finetuned/val_dataset")
}

In [66]:
df = data["train"].to_pandas()
df.head()

,lhs,rhs,is_approx,is_genuine,text,labels,input_ids,attention_mask
0,TailNum,ArrDel15,True,False,You are an expert in relational database theor...,"[128000, 2675, 527, 459, 6335, 304, 72283, 472...","[128000, 2675, 527, 459, 6335, 304, 72283, 472...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
1,client_id,account_id,False,False,You are an expert in relational database theor...,"[128000, 2675, 527, 459, 6335, 304, 72283, 472...","[128000, 2675, 527, 459, 6335, 304, 72283, 472...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
2,emp_HireDate,emp_Region,False,False,You are an expert in relational database theor...,"[128000, 2675, 527, 459, 6335, 304, 72283, 472...","[128000, 2675, 527, 459, 6335, 304, 72283, 472...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
3,client_id,payments,False,False,You are an expert in relational database theor...,"[128000, 2675, 527, 459, 6335, 304, 72283, 472...","[128000, 2675, 527, 459, 6335, 304, 72283, 472...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
4,fullGivenName,lastseason,False,False,You are an expert in relational database theor...,"[128000, 2675, 527, 459, 6335, 304, 72283, 472...","[128000, 2675, 527, 459, 6335, 304, 72283, 472...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."


In [ ]:
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

args = TrainingArguments(
    output_dir="qwen-fd-finetuned",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    evaluation_strategy="steps",
    eval_steps=50,
    logging_steps=10,
    save_steps=100,
    num_train_epochs=3,
    learning_rate=2e-5,
    fp16=True,
    save_total_limit=1
)

In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=data["train"],
    eval_dataset=data["validation"],
    data_collator=collator
)

trainer.train()
model.save_pretrained("qwen-fd-finetuned")
tokenizer.save_pretrained("qwen-fd-finetuned")